# MERA benchmark: Stages 1 and 2

**Goal:** a trustworthy MERA energy evaluation for the critical transverse-field Ising chain, `H = -J sum Z_i Z_{i+1} - h sum X_i` with `J = h = 1`, periodic boundary.

- **Stage 1:** exact reference energy (free-fermion formula), checked against brute-force exact diagonalization (ED).
- **Stage 2:** a finite binary MERA (`L = 2^n`), its energy via Hamiltonian coarse-graining (ascending superoperators), checked against a brute-force state-vector evaluation of the same MERA.

Stage 3 (Evenbly-Vidal optimizer) and Stage 4 (benchmark harness) build on this only after the checks below pass.
CPU only: NumPy plus `opt_einsum`. Install if needed: `pip install numpy scipy opt_einsum matplotlib`.

In [1]:
import itertools, time
import numpy as np
import opt_einsum as oe
import scipy.sparse as sp
import scipy.sparse.linalg as sla
from scipy.linalg import eigh

np.set_printoptions(precision=6, suppress=True)

## Stage 1: exact reference

For the periodic chain the ground state lives in the even-parity sector, where Jordan-Wigner gives antiperiodic momenta `k = pi(2m+1)/L` and

`E0 = - sum_k sqrt(J^2 + h^2 - 2 J h cos k)`.

At criticality `E0 / L -> -4/pi` as `L -> infinity`. This is cheap for any `L`, so it is the reference for every size (including the "exact diagonalization overflows" case of the slides).

In [2]:
def exact_energy_ising(L, J=1.0, h=1.0):
    k = np.pi * (2 * np.arange(L) + 1) / L
    return -np.sum(np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k)))


def ed_energy_ising(L, J=1.0, h=1.0):
    # brute-force sparse ED, only for small L (used purely to validate the formula)
    X = sp.csr_matrix([[0, 1], [1, 0]], dtype=float)
    Z = sp.csr_matrix([[1, 0], [0, -1]], dtype=float)
    I = sp.identity(2, format="csr")

    def op(o, i):
        out = None
        for s in range(L):
            m = o if s == i else I
            out = m if out is None else sp.kron(out, m, format="csr")
        return out

    H = sp.csr_matrix((2**L, 2**L))
    for i in range(L):
        H = H - J * (op(Z, i) @ op(Z, (i + 1) % L)) - h * op(X, i)
    return sla.eigsh(H, k=1, which="SA")[0][0]


print(f"{'L':>4} {'exact formula':>16} {'ED':>16} {'abs diff':>10}")
for L in (4, 6, 8, 10, 12):
    ex, ed = exact_energy_ising(L), ed_energy_ising(L)
    print(f"{L:>4} {ex:>16.10f} {ed:>16.10f} {abs(ex-ed):>10.1e}")
    assert abs(ex - ed) < 1e-8

print()
for L in (16, 64, 256, 1024):
    print(f"L={L:>5}  E0/L = {exact_energy_ising(L)/L:.8f}   (-4/pi = {-4/np.pi:.8f})")

   L    exact formula               ED   abs diff
   4    -5.2262518595    -5.2262518595    8.9e-16
   6    -7.7274066103    -7.7274066103    8.9e-16
   8   -10.2516617910   -10.2516617910    1.8e-15
  10   -12.7849064430   -12.7849064430    7.1e-15


  12   -15.3225951511   -15.3225951511    3.6e-15

L=   16  E0/L = -1.27528715   (-4/pi = -1.27323954)
L=   64  E0/L = -1.27336739   (-4/pi = -1.27323954)
L=  256  E0/L = -1.27324753   (-4/pi = -1.27323954)
L= 1024  E0/L = -1.27324004   (-4/pi = -1.27323954)


## Stage 2: binary MERA

**Layout of one layer** (ring of `N` sites, `N` even): isometries `w_j` act on sites `(2j, 2j+1)` and map two sites to one coarse site of dimension `chi_out = min(chi, D^2)`. Disentanglers `u_j` act on sites `(2j+1, 2j+2 mod N)`, straddling the boundary between neighbouring isometries, and are applied *before* the isometries when coarse-graining.

Layers repeat `L -> L/2 -> ... -> 2` sites; the top "tensor" is a state on the last 2 sites.

**Conventions** (all tensors real):
- `U[a,b,c,d]`: matrix `(ab) x (cd)`, orthogonal. Upper legs `(a,b)`, lower legs `(c,d)`.
- `W[x,p,q]`: matrix `x x (pq)` with orthonormal rows (`W W^T = 1`).
- State going down: `|fine> = U^T W^T |coarse>`. Operators go up: `O' = W (U O U^T) W^T`.

**Energy by coarse-graining:** every Hamiltonian term is ascended layer by layer, terms that land on the same sites are summed, and at the top the energy is the lowest eigenvalue of the 2-site coarse Hamiltonian (that eigenvector is the optimal top state). The ascend routine builds the `einsum` for any operator on any set of sites, with identities inserted automatically where a `u` or `w` only partly overlaps the operator, and `opt_einsum` picks the contraction order.

In [3]:
def random_orthogonal(n, rng):
    q, r = np.linalg.qr(rng.normal(size=(n, n)))
    return q * np.sign(np.diag(r))


def random_isometry(d_in, d_out, rng):
    # returns W with shape (d_out, d_in) and orthonormal rows (W @ W.T = I)
    q, _ = np.linalg.qr(rng.normal(size=(d_in, d_out)))
    return q.T


class MERA:
    """Finite binary MERA on L = 2^n sites (periodic), layer-dependent tensors."""

    def __init__(self, L, chi, d=2, seed=0):
        assert L >= 4 and (L & (L - 1)) == 0, "L must be a power of two, >= 4"
        rng = np.random.default_rng(seed)
        self.L, self.chi, self.d = L, chi, d
        self.n_layers = int(np.log2(L)) - 1          # L -> ... -> 2 sites
        self.dims = [d]                               # site dimension before each layer
        self.U, self.W = [], []
        N, D = L, d
        for _ in range(self.n_layers):
            D_out = min(chi, D * D)
            self.U.append([random_orthogonal(D * D, rng).reshape(D, D, D, D) for _ in range(N // 2)])
            self.W.append([random_isometry(D * D, D_out, rng).reshape(D_out, D, D) for _ in range(N // 2)])
            N //= 2
            D = D_out
            self.dims.append(D)

    def check_constraints(self):
        worst = 0.0
        for k in range(self.n_layers):
            D = self.dims[k]
            for u in self.U[k]:
                m = u.reshape(D * D, D * D)
                worst = max(worst, np.abs(m @ m.T - np.eye(D * D)).max())
            for w in self.W[k]:
                m = w.reshape(w.shape[0], D * D)
                worst = max(worst, np.abs(m @ m.T - np.eye(m.shape[0])).max())
        return worst


m = MERA(16, chi=4)
print("dims per layer:", m.dims, "| layers:", m.n_layers)
print("max violation of unitarity/isometry:", m.check_constraints())

dims per layer: [2, 4, 4, 4] | layers: 3
max violation of unitarity/isometry: 8.881784197001252e-16


In [4]:
def ascend(op_sites, op_tensor, U, W, N):
    """Ascend one operator through one layer.

    op_sites : tuple of distinct sites on a ring of N sites
    op_tensor: shape (D,)*2m, row legs for each site (in op_sites order) then column legs
    returns  : (coarse_sites sorted tuple, tensor with the same row/column layout)
    """
    cnt = itertools.count()
    new = lambda: next(cnt)

    sites = list(op_sites)
    R = {s: new() for s in sites}
    C = {s: new() for s in sites}
    ops = [(op_tensor, [R[s] for s in sites] + [C[s] for s in sites])]
    support = set(sites)

    # disentanglers u_j on sites (2j+1, 2j+2 mod N): O -> U O U^T
    for j in range(N // 2):
        pair = ((2 * j + 1) % N, (2 * j + 2) % N)
        if not (set(pair) & support):
            continue
        up_r, up_c, low_r, low_c = [], [], [], []
        for s in pair:
            up_r.append(new()); up_c.append(new())
            if s in support:
                low_r.append(R[s]); low_c.append(C[s])
            else:                                   # identity on a site the operator does not touch
                z = new(); low_r.append(z); low_c.append(z)
        ops.append((U[j], up_r + low_r))
        ops.append((U[j], up_c + low_c))
        for s, ur, uc in zip(pair, up_r, up_c):
            R[s], C[s] = ur, uc
            support.add(s)

    # isometries w_j on sites (2j, 2j+1): O -> W O W^T
    coarse = {}
    for j in range(N // 2):
        pair = (2 * j, 2 * j + 1)
        if not (set(pair) & support):
            continue
        xr, xc = new(), new()
        low_r, low_c = [], []
        for s in pair:
            if s in support:
                low_r.append(R[s]); low_c.append(C[s])
            else:
                z = new(); low_r.append(z); low_c.append(z)
        ops.append((W[j], [xr] + low_r))
        ops.append((W[j], [xc] + low_c))
        coarse[j] = (xr, xc)

    cs = sorted(coarse)
    out = [coarse[j][0] for j in cs] + [coarse[j][1] for j in cs]
    args = []
    for t, idx in ops:
        args += [t, idx]
    return tuple(cs), oe.contract(*args, out, optimize="auto")


def ising_bond_terms(L, J=1.0, h=1.0):
    X = np.array([[0.0, 1.0], [1.0, 0.0]])
    Z = np.diag([1.0, -1.0])
    I = np.eye(2)
    # onsite field split evenly over the two bonds touching each site
    hb = -J * np.kron(Z, Z) - 0.5 * h * (np.kron(X, I) + np.kron(I, X))
    hb = hb.reshape(2, 2, 2, 2)
    terms = []
    for i in range(L):
        j = (i + 1) % L
        if i < j:
            terms.append(((i, j), hb))
        else:                                        # wrap-around bond: sort sites, swap legs
            terms.append(((j, i), hb.transpose(1, 0, 3, 2)))
    return terms


def mera_energy(mera, terms):
    """Energy by ascending all terms to the top, then diagonalizing the top Hamiltonian.
    Returns (energy, top_state_vector)."""
    ops = {}
    for s, t in terms:
        ops[s] = ops[s] + t if s in ops else t
    N = mera.L
    for k in range(mera.n_layers):
        nxt = {}
        for sites, t in ops.items():
            s2, t2 = ascend(sites, t, mera.U[k], mera.W[k], N)
            nxt[s2] = nxt[s2] + t2 if s2 in nxt else t2
        ops, N = nxt, N // 2
    assert N == 2
    D = mera.dims[-1]
    Htop = np.zeros((D * D, D * D))
    for sites, t in ops.items():
        mat = t.reshape(D ** len(sites), D ** len(sites))
        if sites == (0, 1):
            Htop += mat
        elif sites == (0,):
            Htop += np.kron(mat, np.eye(D))
        elif sites == (1,):
            Htop += np.kron(np.eye(D), mat)
        else:
            raise ValueError(sites)
    evals, evecs = eigh(Htop)
    return evals[0], evecs[:, 0]

### Independent check: build the full state vector

To be sure the coarse-graining is right, the same MERA is also evaluated the brute-force way for small `L`: start from the top state, apply `W^T` (1 site -> 2 sites) and then `U^T` on every pair, layer by layer, until a `2^L` state vector exists, then compute `<H>` directly. The two numbers must agree to machine precision for *random* tensors, which tests the contraction itself (not the quality of the MERA).

In [5]:
def mera_state(mera, top_vec):
    D = mera.dims[-1]
    psi = top_vec.reshape(D, D)
    N = 2
    for k in reversed(range(mera.n_layers)):
        # coarse -> fine: each coarse leg x becomes (p, q) through W[x, p, q]
        for j in range(N):
            psi = np.tensordot(psi, mera.W[k][j], axes=([2 * j], [0]))
            psi = np.moveaxis(psi, [-2, -1], [2 * j, 2 * j + 1])
        N *= 2
        # disentanglers: state[c, d] = sum_ab U[a, b, c, d] state[a, b]
        for j in range(N // 2):
            s1, s2 = (2 * j + 1) % N, (2 * j + 2) % N
            psi = np.tensordot(psi, mera.U[k][j], axes=([s1, s2], [0, 1]))
            psi = np.moveaxis(psi, [-2, -1], [s1, s2])
    return psi


def state_energy(psi, J=1.0, h=1.0):
    L = psi.ndim
    norm = np.sum(psi**2)
    z = np.array([1.0, -1.0])
    p = psi**2
    E = 0.0
    for i in range(L):
        j = (i + 1) % L
        shape_i = [1] * L; shape_i[i] = 2
        shape_j = [1] * L; shape_j[j] = 2
        zz = np.sum(p * z.reshape(shape_i) * z.reshape(shape_j))
        x = np.sum(psi * np.flip(psi, axis=i))
        E += -J * zz - h * x
    return E / norm, norm


print(f"{'L':>3} {'chi':>3} {'seed':>4} {'E (coarse-grain)':>18} {'E (full state)':>18} {'|diff|':>9} {'norm':>8}")
worst = 0.0
for L in (4, 8, 16):
    for chi in (2, 3, 4, 6):
        for seed in (0, 1):
            m = MERA(L, chi, seed=seed)
            E_cg, top = mera_energy(m, ising_bond_terms(L))
            E_sv, norm = state_energy(mera_state(m, top))
            worst = max(worst, abs(E_cg - E_sv))
            print(f"{L:>3} {chi:>3} {seed:>4} {E_cg:>18.12f} {E_sv:>18.12f} {abs(E_cg-E_sv):>9.1e} {norm:>8.6f}")
print("\nworst disagreement:", worst)
assert worst < 1e-10, "coarse-graining does not match the brute-force state"
print("PASS: coarse-grained MERA energy matches the full state-vector energy.")

  L chi seed   E (coarse-grain)     E (full state)    |diff|     norm
  4   2    0    -1.567256327432    -1.567256327432   8.9e-16 1.000000
  4   2    1    -2.499964639744    -2.499964639744   1.1e-14 1.000000
  4   3    0    -2.501776315043    -2.501776315043   4.4e-16 1.000000
  4   3    1    -4.244255443229    -4.244255443229   9.8e-15 1.000000
  4   4    0    -5.226251859506    -5.226251859506   7.1e-15 1.000000
  4   4    1    -5.226251859505    -5.226251859506   1.4e-14 1.000000
  4   6    0    -5.226251859506    -5.226251859506   7.1e-15 1.000000
  4   6    1    -5.226251859505    -5.226251859506   1.4e-14 1.000000


  8   2    0    -1.760598827573    -1.760598827573   1.3e-15 1.000000
  8   2    1    -1.776250648473    -1.776250648473   4.4e-16 1.000000
  8   3    0    -2.293859377714    -2.293859377714   2.7e-15 1.000000
  8   3    1    -1.572383401186    -1.572383401186   3.8e-15 1.000000
  8   4    0    -2.403902214904    -2.403902214904   2.7e-15 1.000000


  8   4    1    -2.760479355040    -2.760479355040   4.4e-16 1.000000


  8   6    0    -3.284494970973    -3.284494970973   4.4e-16 1.000000
  8   6    1    -3.318851646988    -3.318851646988   2.2e-15 1.000000


 16   2    0    -1.106533515056    -1.106533515056   4.9e-15 1.000000


 16   2    1    -0.449792298695    -0.449792298695   3.3e-16 1.000000
 16   3    0    -1.400873314799    -1.400873314799   2.7e-15 1.000000


 16   3    1    -1.203719728632    -1.203719728632   4.2e-15 1.000000


 16   4    0    -0.448980551159    -0.448980551159   5.6e-16 1.000000


 16   4    1    -0.916888067486    -0.916888067486   2.2e-16 1.000000
 16   6    0    -0.943078265672    -0.943078265672   3.3e-16 1.000000


 16   6    1    -0.634661244875    -0.634661244875   1.0e-15 1.000000

worst disagreement: 1.4210854715202004e-14
PASS: coarse-grained MERA energy matches the full state-vector energy.


### Sanity: energies are variational

Random MERAs are bad states, so their energy must sit *above* the exact ground state. (If a random MERA ever came out below, the contraction would be wrong.) The optimizer in Stage 3 should push the energy per site down towards `-4/pi`.

In [6]:
print(f"{'L':>4} {'chi':>3} {'E_random/L':>12} {'E_exact/L':>12}")
for L in (8, 16, 32):
    for chi in (2, 4):
        m = MERA(L, chi, seed=3)
        E, _ = mera_energy(m, ising_bond_terms(L))
        Ex = exact_energy_ising(L)
        assert E > Ex - 1e-10
        print(f"{L:>4} {chi:>3} {E/L:>12.6f} {Ex/L:>12.6f}")

   L chi   E_random/L    E_exact/L
   8   2    -0.160380    -1.281458


   8   4    -0.215582    -1.281458


  16   2     0.052746    -1.275287
  16   4    -0.079801    -1.275287


  32   2     0.045461    -1.273751


  32   4    -0.013043    -1.273751


### Cost of one energy evaluation (random MERA, CPU)

This is a first look at the time per evaluation before the optimizer exists. It does not yet exploit translation invariance, so it is an upper bound on what the optimized benchmark will need.

In [7]:
print(f"{'L':>4} {'chi':>3} {'time (s)':>9}")
for L, chi in [(32, 4), (64, 4), (32, 8), (64, 8)]:
    m = MERA(L, chi, seed=0)
    t0 = time.perf_counter()
    E, _ = mera_energy(m, ising_bond_terms(L))
    print(f"{L:>4} {chi:>3} {time.perf_counter()-t0:>9.2f}")

   L chi  time (s)


  32   4      0.09


  64   4      0.20


  32   8      0.58


  64   8      1.54


## Status

- Stage 1 done: exact reference verified against ED (`E0/L -> -4/pi`).
- Stage 2 done: MERA structure plus coarse-grained energy, verified against brute-force state vectors to ~1e-12.

**Next (Stage 3):** Evenbly-Vidal updates. Each update contracts the *environment* of a tensor (the same network as the energy with that tensor removed), takes an SVD, and replaces the tensor by `-U V^T`; sweep `u`, `w`, and the top state until the energy converges.